In [17]:
import sys
sys.path.append("../../")

import os
import numpy as np
import torch

import grid_pkg
import controller_pkg
import controller_utils
import dQPTH
import evals

torch.set_default_dtype(torch.double)
torch.set_printoptions(threshold=10000)
NUMPY_SEED = 0
TORCH_SEED = 0
np.random.seed(NUMPY_SEED)
torch.manual_seed(TORCH_SEED)

In [ ]:
line_data_loc = '../case118_line_data.pt'
bus_data_loc = '../case118_bus_data.pt'
gen_data_loc = '../case118_gen_data.pt'
ptdf_data_loc = '../case118_ptdf_data.pt'

bus_with_curt = torch.load(gen_data_loc, weights_only=True)[:,0].int()
num_curt = bus_with_curt.shape[0]
bus_with_batt = torch.tensor([10*i+2 for i in range(12)], dtype=torch.int)
num_batt = bus_with_batt.shape[0]
delta_t = 15

grid = grid_pkg.Grid(bus_with_curt, bus_with_batt, delta_t, line_data_loc, bus_data_loc, gen_data_loc, ptdf_data_loc)
num_agents = 3
batt_cost = 100
curt_change_cost = 0.01
curt_net_cost = 1
bus_slack_cost = 1e8
line_slack_cost = 1e2

num_agents = 3
nodes_1 = [i for i in range(0, 42)] + [112, 113, 114, 116]
nodes_2 = [i for i in range(42, 69)] 
nodes_3 = [i for i in range(69, 112)] + [115, 117]
initial_state = grid.state

bus_idx_gap = 10
T = 20
H = 5
file_path = "../tauxDeChargeMTJLMA2juillet2018.txt" # data is from https://github.com/rtechair/ZonalControlSimulator

noise_mags = [2.0, 2.4, 2.45, 2.5, 3.0, 3.5, 4.0]
offset_vals = [30 * i for i in range(10)]
forecast_skews = dict()

for noise_mag in noise_mags:
    for offset in offset_vals:
        forecast_skews[(noise_mag, offset)] = controller_utils.get_RTE_noise_values(file_path, grid, T, H, noise_mag, bus_idx_gap, offset=offset)
forecast_skews["noise_mags"] = noise_mags
forecast_skews["offset_vals"] = offset_vals

torch.save(forecast_skews, "forecast_skews.pt")

In [ ]:
num_forecast_traj = 10
num_forecast_seeds = 5
forecast_seeds = [i for i in range(num_forecast_seeds)]
forecast_radius = 0.2
forecast_trajs = dict()

for noise_mag in noise_mags:
    for offset in offset_vals:
        forecast_skew = forecast_skews[(noise_mag,offset)]
        for forecast_seed in forecast_seeds:
            torch.manual_seed(forecast_seed)
            all_forecast_traj = torch.zeros(num_forecast_traj, T+H, grid.num_buses)
            for i in range(num_forecast_traj):
                dist_error = torch.rand_like(forecast_skew) * forecast_radius * 2 + (1-forecast_radius)
                all_forecast_traj[i] = dist_error * forecast_skew
            forecast_trajs[(noise_mag, offset, forecast_seed)] = all_forecast_traj
        forecast_trajs[(noise_mag, offset)] = forecast_skew
forecast_trajs["num_forecast_traj"] = num_forecast_traj
forecast_trajs["num_forecast_seeds"] = num_forecast_seeds
forecast_trajs["forecast_radius"] = forecast_radius

torch.save(forecast_trajs, f"forecast_trajs_rad{forecast_radius}.pt")

In [ ]:
num_test_traj = 10
test_trajs_no_mismatch = dict()
test_radius = 0.2
for noise_mag in noise_mags:
    for offset in offset_vals:
        forecast_skew = forecast_skews[(noise_mag,offset)]
        all_test_traj = torch.zeros(num_test_traj, T+H, grid.num_buses)
        for i in range(num_test_traj):
            dist_error = torch.rand_like(forecast_skew) * test_radius * 2 + (1-test_radius)
            all_test_traj[i] = dist_error * forecast_skew  # test_skew == 0 => forecast_skew == test_skew
        test_trajs_no_mismatch[(noise_mag, offset)] = all_test_traj
test_trajs_no_mismatch["num_test_traj"] = num_test_traj
test_trajs_no_mismatch["test_radius"] = test_radius

torch.save(test_trajs_no_mismatch, f"test_trajs_no_mismatch_rad{test_radius}.pt")

In [ ]:
# split into smaller files since one file may be too large
num_test_traj = 10
test_radius = 0.2
test_skew_mags = [10, 20, 30, 40]
num_test_seeds = 3
test_seeds = [i for i in range(num_test_seeds)]
folder = f"test_trajs_with_mismatch_rad{test_radius}"
os.makedirs(folder, exist_ok=True)
for noise_mag in noise_mags:
    for offset in offset_vals:
        forecast_skew = forecast_skews[(noise_mag,offset)]
        for test_skew_mag in test_skew_mags:
            test_trajs_with_mismatch = dict()
            for test_seed in test_seeds:
                torch.manual_seed(test_seed)
                dist_error = torch.rand_like(forecast_skew) * (test_skew_mag / 100) * 2 + (1-test_skew_mag/100)
                test_skew = forecast_skew * dist_error
                all_test_traj = torch.zeros(num_test_traj, T+H, grid.num_buses)
                for i in range(num_test_traj):
                    dist_error = torch.rand_like(test_skew) * test_radius * 2 + (1-test_radius)
                    all_test_traj[i] = dist_error * test_skew
                test_trajs_with_mismatch[(noise_mag,offset,test_skew_mag,test_seed)] = all_test_traj
            test_trajs_with_mismatch["num_test_traj"] = num_test_traj
            test_trajs_with_mismatch["test_radius"] = test_radius
            test_trajs_with_mismatch["test_skew_mags"] = test_skew_mags
            test_trajs_with_mismatch["num_test_seeds"] = num_test_seeds
            torch.save(test_trajs_with_mismatch, f"{folder}/test_trajs_with_mismatch_noise_mag{noise_mag}_offset{offset}_tsm{test_skew_mag}.pt")
